# Run the backend on Colab GPU

Runs this repo's FastAPI backend (`backend/`) on a Colab GPU runtime and exposes it to the
internet with a Cloudflare quick tunnel, so your **local** frontend (`npm run dev` on your own
machine, unchanged) can call it instead of a slow local CPU backend.

**Before running:**
1. `Runtime > Change runtime type > GPU`, then `Runtime > Restart session` if you changed it.
2. One-time only: upload `backend/models/best_augmented.pt` and everything in `backend/data/`
   (`class_keywords.json`, `embeddings.json`, ...) from your machine into a Google Drive folder
   at `MyDrive/lipton-sku-classifier-assets/models/` and `MyDrive/lipton-sku-classifier-assets/data/`
   respectively. These files are gitignored (see `backend/.gitignore`) so they don't come from
   `git clone` below — Drive is just the hand-off point between your machine and the Colab VM.
3. Run every cell top to bottom. The last cell prints the URL to paste into your local
   `frontend/.env` as `VITE_API_BASE`.

The backend and tunnel keep running as background processes as long as this notebook's runtime
stays alive — closing the tab eventually disconnects the runtime (Colab free tier), which kills
both, so re-run the notebook to get a new session and a new tunnel URL.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU, restart, and re-run."
    )

In [ ]:
REPO_URL = "https://github.com/ansshahzadd/lipton-sku-classifier.git"
REPO_DIR = "/content/lipton-sku-classifier"
BACKEND_DIR = f"{REPO_DIR}/backend"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

In [ ]:
# Install everything from requirements.txt, then swap the CPU paddlepaddle build for the
# GPU one so OCR can run on the GPU too (see OCR_DEVICE below).
%pip install -q -r {BACKEND_DIR}/requirements.txt
%pip uninstall -y -q paddlepaddle
%pip install -q paddlepaddle-gpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

ASSETS_DIR = "/content/drive/MyDrive/lipton-sku-classifier-assets"

for sub in ("models", "data"):
    src = os.path.join(ASSETS_DIR, sub)
    dst = os.path.join(BACKEND_DIR, sub)
    if not os.path.isdir(src):
        raise FileNotFoundError(
            f"Expected {src} on your Drive. Upload your {sub}/ files there first (see the "
            "instructions in the first cell), then re-run this cell."
        )
    os.makedirs(dst, exist_ok=True)
    for fname in os.listdir(src):
        shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
    print(f"{sub}/ ->", os.listdir(dst))

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
import re, subprocess, threading, time

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
_url_pattern = re.compile(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com")

def _watch_tunnel_output():
    global public_url
    for line in tunnel_proc.stdout:
        match = _url_pattern.search(line)
        if match and public_url is None:
            public_url = match.group(0)
        print(line, end="")

threading.Thread(target=_watch_tunnel_output, daemon=True).start()

for _ in range(60):
    if public_url:
        break
    time.sleep(1)

if not public_url:
    raise RuntimeError("No tunnel URL after 60s — check the cloudflared output above for errors.")

print("\nTunnel URL:", public_url)

In [ ]:
import os, subprocess, threading

env = os.environ.copy()
env["PUBLIC_BASE_URL"] = public_url
env["FRONTEND_ORIGINS"] = "http://localhost:5173,http://127.0.0.1:5173"
env["OCR_DEVICE"] = "gpu"  # set to "cpu" here if paddlepaddle-gpu errors on this GPU/CUDA combo

backend_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=BACKEND_DIR,
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

def _watch_backend_output():
    for line in backend_proc.stdout:
        print(line, end="")

threading.Thread(target=_watch_backend_output, daemon=True).start()
print("uvicorn starting (pid", backend_proc.pid, ") — first request will be slow while models load.")

In [ ]:
import time, urllib.request

up = False
for _ in range(90):
    try:
        with urllib.request.urlopen(public_url + "/api/dashboard", timeout=3) as resp:
            if resp.status == 200:
                up = True
                break
    except Exception:
        pass
    time.sleep(2)

if not up:
    print("Backend didn't respond in time — check the uvicorn logs in the cell above for errors.")
else:
    print("Backend is up and reachable through the tunnel.\n")

print("=" * 60)
print("On your LOCAL machine, set this in frontend/.env, then `npm run dev`:")
print(f"VITE_API_BASE={public_url}")
print("=" * 60)

## Notes

- If `paddlepaddle-gpu` fails to use the GPU (CUDA version mismatch is the usual cause), set
  `OCR_DEVICE = "cpu"` in the uvicorn cell above and re-run just that cell + the health-check
  cell — you'll still get the YOLO + DINOv3 GPU speedup, just not for OCR.
- Cloudflare's quick tunnel URL is random and changes every time you re-run the tunnel cell —
  update `VITE_API_BASE` locally whenever you restart this notebook.
- To stop everything: `backend_proc.terminate(); tunnel_proc.terminate()`, or just stop/disconnect
  the Colab runtime.